# Notebook 1: PyTDC Drug-Drug Interaction Models

Trains and evaluates two models on the **same DDI DrugBank dataset**:

| # | Dataset | Task | Method |
|---|---------|------|--------|
| A | DDI DrugBank | **Multi-class** (89 interaction mechanism types) | Morgan FP + Random Forest |
| B | DDI DrugBank | **Multi-class** (top-5 interaction types) | Molecular GNN (GCN + Morgan FP) |

Both models are trained on the DrugBank DDI dataset for direct comparison in Notebook 3.

All models saved to Google Drive at the end.


In [ ]:
# ─── Install Dependencies ───────────────────────────────────────────────────
!pip install -q 'numpy<2' rdkit scikit-learn PyTDC matplotlib seaborn
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

In [ ]:
!pip install scikit-learn==1.2.2

In [ ]:
# ─── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, json, gc
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINConv, global_add_pool

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')

In [ ]:
# ─── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/drug_interaction_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

---
## Shared Utilities

In [ ]:
# ─── Morgan Fingerprint (used by DDI RF model) ────────────────────────────────
def smiles_to_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

def build_pair_features(df, col1='Drug1', col2='Drug2'):
    fp1 = np.stack(df[col1].apply(smiles_to_fp).values)
    fp2 = np.stack(df[col2].apply(smiles_to_fp).values)
    return np.hstack([fp1, fp2])

# ─── Atom features for GNN (used by TWOSIDES GNN) ────────────────────────────
ATOM_TYPES = ['C','N','O','S','F','Cl','Br','I','P','Si','B','Se','other']
N_ATOM_FEATS = len(ATOM_TYPES) + 4  # = 17

def atom_features(atom):
    sym = atom.GetSymbol()
    idx = ATOM_TYPES.index(sym) if sym in ATOM_TYPES else len(ATOM_TYPES) - 1
    oh  = [0.0] * len(ATOM_TYPES); oh[idx] = 1.0
    return oh + [
        atom.GetDegree() / 6.0,
        atom.GetFormalCharge() / 4.0,
        float(atom.GetIsAromatic()),
        atom.GetTotalNumHs() / 4.0,
    ]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i,j],[j,i]]
    if not edges:
        edges = [[0,0]]
    ei = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=ei)

print('Utilities ready')

---
# Part 1 — DDI Multi-class Classification (DrugBank)

**Task**: Given SMILES of two drugs, predict which of the **89 interaction mechanism types** applies.  
**Dataset**: 191,808 pairs, 1,706 drugs, pre-split by PyTDC.  
**Label map**: fetched live via `get_label_map(name='DrugBank', task='DDI')` — no hardcoding.

In [ ]:
# ─── 1.1  Load DDI Dataset ───────────────────────────────────────────────────
from tdc.multi_pred import DDI
from tdc.utils import get_label_map

ddi_data  = DDI(name='DrugBank')
split     = ddi_data.get_split()
train_ddi = split['train'].copy()
val_ddi   = split['valid'].copy()
test_ddi  = split['test'].copy()

print(f'Train: {len(train_ddi)} | Val: {len(val_ddi)} | Test: {len(test_ddi)}')
print(train_ddi.head())

In [ ]:
# ─── 1.2  Load Official Label Map ────────────────────────────────────────────
# get_label_map returns {int_id: description_string} for all 89 types
DDI_LABEL_MAP = get_label_map(name='DrugBank', task='DDI')
print(f'Label map contains {len(DDI_LABEL_MAP)} interaction types.')
print('Sample entries:')
for k in list(DDI_LABEL_MAP.keys())[:4]:
    print(f'  {k}: {DDI_LABEL_MAP[k]}')

# Persist for notebook 3
with open('/content/ddi_label_map.json', 'w') as f:
    json.dump({str(k): v for k, v in DDI_LABEL_MAP.items()}, f)

In [ ]:
# ─── 1.3  Inspect & Visualise Class Distribution ─────────────────────────────
vc = train_ddi['Y'].value_counts()
print(f'Unique interaction types in train: {len(vc)}')
print('Top 10 by frequency:')
print(vc.head(10))

top20       = vc.head(20)
top20_names = [DDI_LABEL_MAP.get(int(k), str(k))[:65] for k in top20.index]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(top20_names[::-1], top20.values[::-1], color='steelblue')
ax.set_xlabel('Count in Training Set')
ax.set_title('DDI DrugBank — Top 20 Interaction Types (Train)')
plt.tight_layout()
plt.savefig('/content/ddi_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 1.4  Encode Integer Labels (1-indexed → 0-indexed) ──────────────────────
le = LabelEncoder()
le.fit(train_ddi['Y'].values)
known_classes = set(le.classes_)

def safe_encode(le, series):
    """Map unseen labels to the first known class to avoid transform errors."""
    arr = series.map(lambda v: v if v in known_classes else le.classes_[0]).values
    return le.transform(arr)

y_train_ddi = le.transform(train_ddi['Y'].values)
y_val_ddi   = safe_encode(le, val_ddi['Y'])
y_test_ddi  = safe_encode(le, test_ddi['Y'])

n_classes_ddi = len(le.classes_)
print(f'Number of DDI classes: {n_classes_ddi}')

with open('/content/ddi_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

In [ ]:
# ─── 1.5  Build Fingerprint Features ─────────────────────────────────────────
# 191k pairs × 4096 features ≈ 3 GB float32 — fits in Colab RAM for DDI.
# (TWOSIDES at 4.6M pairs would NOT fit — hence the GNN approach there.)
print('Building Morgan fingerprint pair features for DDI...')
print('Expected time: 5-10 minutes on Colab CPU.')

X_train_ddi = build_pair_features(train_ddi)
X_val_ddi   = build_pair_features(val_ddi)
X_test_ddi  = build_pair_features(test_ddi)

print(f'Train: {X_train_ddi.shape} | Val: {X_val_ddi.shape} | Test: {X_test_ddi.shape}')

In [ ]:
# ─── 1.6  Train Multi-class Random Forest ────────────────────────────────────
ddi_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_leaf=1,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
print('Training DDI multi-class classifier (89 types)...')
ddi_model.fit(X_train_ddi, y_train_ddi)
print('Done.')

In [ ]:
# ─── 1.7  Evaluation Helper & Results ────────────────────────────────────────
def eval_multiclass(model, X, y_true, split_name):
    y_pred  = model.predict(X)
    acc     = accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    f1_mic  = f1_score(y_true, y_pred, average='micro',    zero_division=0)
    f1_wt   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'\n=== {split_name} ===')
    print(f'  Accuracy    : {acc:.4f}')
    print(f'  Macro F1    : {f1_mac:.4f}')
    print(f'  Micro F1    : {f1_mic:.4f}')
    print(f'  Weighted F1 : {f1_wt:.4f}')
    return dict(split=split_name, accuracy=acc,
                macro_f1=f1_mac, micro_f1=f1_mic, weighted_f1=f1_wt,
                y_pred=y_pred)

ddi_results = {
    'train': eval_multiclass(ddi_model, X_train_ddi, y_train_ddi, 'Train'),
    'val':   eval_multiclass(ddi_model, X_val_ddi,   y_val_ddi,   'Validation'),
    'test':  eval_multiclass(ddi_model, X_test_ddi,  y_test_ddi,  'Test'),
}

In [ ]:
# ─── 1.8  Metrics Bar Chart ───────────────────────────────────────────────────
met = pd.DataFrame([{k:v for k,v in r.items()
                     if k in ('split','accuracy','macro_f1','micro_f1','weighted_f1')}
                    for r in ddi_results.values()])
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(met)); w = 0.2
ax.bar(x-1.5*w, met['accuracy'],    w, label='Accuracy',    color='steelblue')
ax.bar(x-0.5*w, met['macro_f1'],    w, label='Macro F1',    color='seagreen')
ax.bar(x+0.5*w, met['micro_f1'],    w, label='Micro F1',    color='coral')
ax.bar(x+1.5*w, met['weighted_f1'], w, label='Weighted F1', color='orchid')
ax.set_xticks(x); ax.set_xticklabels(met['split'])
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('DDI Multi-class Model (89 types) — Performance by Split')
ax.legend(); plt.tight_layout()
plt.savefig('/content/ddi_metrics_bar.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 1.9  Per-class F1 — Test Set ────────────────────────────────────────────
per_cls_f1 = f1_score(y_test_ddi, ddi_results['test']['y_pred'],
                      average=None, zero_division=0)

class_labels = [DDI_LABEL_MAP.get(int(le.classes_[i]), str(le.classes_[i]))[:70]
                for i in range(len(le.classes_))]
cls_df = pd.DataFrame({'label': class_labels, 'f1': per_cls_f1})
top25  = cls_df.sort_values('f1', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(14, 9))
ax.barh(top25['label'].values[::-1], top25['f1'].values[::-1], color='teal')
ax.set_xlabel('F1 Score'); ax.set_xlim(0, 1)
ax.set_title('DDI Multi-class — Top 25 Per-class F1 (Test Set)')
plt.tight_layout()
plt.savefig('/content/ddi_per_class_f1.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 1.10  Confusion Matrix (top 15 classes by training frequency) ────────────
top15_raw = train_ddi['Y'].value_counts().head(15).index.tolist()
top15_enc = np.array([le.transform([v])[0] for v in top15_raw if v in known_classes])

mask    = np.isin(y_test_ddi, top15_enc)
yt_sub  = y_test_ddi[mask]
yp_sub  = ddi_results['test']['y_pred'][mask]
uniq    = np.unique(np.concatenate([yt_sub, yp_sub]))
snames  = [DDI_LABEL_MAP.get(int(le.classes_[c]), str(c))[:45] for c in uniq]

cm = confusion_matrix(yt_sub, yp_sub, labels=uniq)
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(cm, xticklabels=snames, yticklabels=snames,
            cmap='Blues', fmt='d', annot=True, linewidths=0.3, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('DDI Multi-class — Confusion Matrix (Top 15 Types, Test)')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/content/ddi_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

---
# Part 2 — DDI Multi-class Classification (GNN)

**Task**: Same as Part 1 — given SMILES of two drugs, predict the interaction mechanism type.  
**Dataset**: Same DDI DrugBank dataset, top-5 most frequent interaction types (500 samples each = 2,500 total).  
**Method**: Molecular GNN — each drug encoded as atom graph + Morgan fingerprint; pair embeddings fed to classifier.  
**Why GNN?**: GNNs capture molecular substructure patterns that flat fingerprints miss, enabling direct comparison.


In [ ]:
# ─── 2.1  Additional GNN Imports ─────────────────────────────────────────────
!pip install -q torch-geometric

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import StratifiedKFold

torch.manual_seed(42)
np.random.seed(42)
print('GNN imports ready.')


In [ ]:
# ─── 2.2  Prepare DDI Dataset for GNN (top-5 interaction classes) ─────────────
# We reuse the same DDI split already loaded above (train_ddi / val_ddi / test_ddi)
# and the same label encoder (le). Here we create a balanced sub-sample using
# the top-5 most frequent interaction types for tractable GNN training.

full_ddi_df = pd.concat([train_ddi, val_ddi, test_ddi], ignore_index=True)

# Top-5 classes by frequency
top5_classes = full_ddi_df['Y'].value_counts().head(5).index
gnn_df = full_ddi_df[full_ddi_df['Y'].isin(top5_classes)].copy()

# Balanced sample: 500 per class
gnn_df = gnn_df.groupby('Y', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 500), random_state=42)
).reset_index(drop=True)

# Encode labels
gnn_le = LabelEncoder()
gnn_df['label_enc'] = gnn_le.fit_transform(gnn_df['Y'])
N_GNN_CLASSES = len(gnn_le.classes_)

# Save GNN label encoder
with open('/content/ddi_gnn_label_encoder.pkl', 'wb') as f:
    pickle.dump(gnn_le, f)

# Build class name mapping for GNN classes
gnn_class_names = [DDI_LABEL_MAP.get(int(c), str(c)) for c in gnn_le.classes_]
gnn_label_map = {i: gnn_class_names[i] for i in range(N_GNN_CLASSES)}
with open('/content/ddi_gnn_label_map.json', 'w') as f:
    json.dump(gnn_label_map, f)

print(f'GNN dataset: {len(gnn_df)} pairs, {N_GNN_CLASSES} classes')
print('Class distribution:')
print(gnn_df['Y'].value_counts())
print('\nClass names:')
for i, name in enumerate(gnn_class_names):
    print(f'  {i}: {name[:80]}')


In [ ]:
# ─── 2.3  SMILES → Molecular Graph ───────────────────────────────────────────
GNN_ATOM_TYPES = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I']
GNN_NODE_FEAT_DIM = len(GNN_ATOM_TYPES) + 3  # one-hot + other + degree + aromaticity
GNN_FP_DIM = 128

def smiles_to_gnn_graph(smiles):
    """Convert SMILES to PyG Data object with atom features + Morgan fingerprint."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    # Node features: atom type one-hot + degree + aromaticity
    node_feats = []
    for atom in mol.GetAtoms():
        one_hot = [int(atom.GetSymbol() == t) for t in GNN_ATOM_TYPES]
        one_hot.append(int(atom.GetSymbol() not in GNN_ATOM_TYPES))  # 'other'
        one_hot += [atom.GetDegree() / 6.0, int(atom.GetIsAromatic())]
        node_feats.append(one_hot)
    x = torch.tensor(node_feats, dtype=torch.float)

    # Edge index
    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Morgan fingerprint (128-bit) as auxiliary feature
    gen = AllChem.GetMorganGenerator(radius=2, fpSize=GNN_FP_DIM)
    fp = torch.tensor(list(gen.GetFingerprint(mol)), dtype=torch.float)

    return Data(x=x, edge_index=edge_index, fp=fp)

# Smoke test
_g = smiles_to_gnn_graph('CC(=O)Oc1ccccc1C(=O)O')
print(f'Test graph — Nodes: {_g.num_nodes}, Edges: {_g.num_edges}, Feat dim: {_g.x.shape[1]}')


In [ ]:
# ─── 2.4  Convert Dataset to Graph Pairs ──────────────────────────────────────
print('Converting SMILES to graphs (this may take a minute)...')
gnn_pairs, gnn_labels = [], []

for _, row in gnn_df.iterrows():
    g1 = smiles_to_gnn_graph(row['Drug1'])
    g2 = smiles_to_gnn_graph(row['Drug2'])
    if g1 is not None and g2 is not None:
        gnn_pairs.append((g1, g2))
        gnn_labels.append(row['label_enc'])

gnn_labels = np.array(gnn_labels)
print(f'{len(gnn_pairs)} valid pairs converted.')


In [ ]:
# ─── 2.5  GNN Model Architecture ─────────────────────────────────────────────
GNN_HIDDEN = 64
GNN_EMBED  = 128

class MolGNN(nn.Module):
    """3-layer GCN that encodes a single molecule into a fixed-size embedding."""
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(GNN_NODE_FEAT_DIM, GNN_HIDDEN)
        self.conv2 = GCNConv(GNN_HIDDEN, GNN_HIDDEN)
        self.conv3 = GCNConv(GNN_HIDDEN, GNN_HIDDEN)
        self.fc = nn.Linear(GNN_HIDDEN + GNN_FP_DIM, GNN_EMBED)

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei))
        x = F.relu(self.conv2(x, ei))
        x = F.relu(self.conv3(x, ei))
        x = global_mean_pool(x, batch)
        fp = data.fp.view(x.size(0), -1)
        return F.relu(self.fc(torch.cat([x, fp], dim=1)))


class DDIGNNModel(nn.Module):
    """Predict interaction type from a pair of molecule embeddings."""
    def __init__(self, n_classes=N_GNN_CLASSES):
        super().__init__()
        self.encoder = MolGNN()
        self.classifier = nn.Sequential(
            nn.Linear(GNN_EMBED * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, mol_a, mol_b):
        emb_a = self.encoder(mol_a)
        emb_b = self.encoder(mol_b)
        return self.classifier(torch.cat([emb_a, emb_b], dim=1))


ddi_gnn = DDIGNNModel().to(DEVICE)
print(f'DDI GNN Parameters: {sum(p.numel() for p in ddi_gnn.parameters()):,}')


In [ ]:
# ─── 2.6  Train & Evaluate GNN (3-fold Cross-validation) ─────────────────────
GNN_N_EPOCHS = 50
GNN_N_FOLDS  = 3
GNN_BATCH    = 32

def make_gnn_batch(pairs, idx):
    g1 = Batch.from_data_list([pairs[i][0] for i in idx]).to(DEVICE)
    g2 = Batch.from_data_list([pairs[i][1] for i in idx]).to(DEVICE)
    return g1, g2

def gnn_train_epoch(model, optimizer, pairs, labels):
    model.train()
    criterion = nn.CrossEntropyLoss()
    idx = np.random.permutation(len(pairs))
    total_loss = 0
    for start in range(0, len(idx), GNN_BATCH):
        batch_idx = idx[start:start + GNN_BATCH]
        g1, g2 = make_gnn_batch(pairs, batch_idx)
        y = torch.tensor(labels[batch_idx], dtype=torch.long).to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(g1, g2), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_idx)
    return total_loss / len(pairs)

@torch.no_grad()
def gnn_evaluate(model, pairs, labels):
    model.eval()
    all_preds, all_probs = [], []
    for start in range(0, len(pairs), 64):
        idx = list(range(start, min(start + 64, len(pairs))))
        g1, g2 = make_gnn_batch(pairs, idx)
        logits = model(g1, g2)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
    return accuracy_score(labels, all_preds), np.array(all_preds), np.array(all_probs)


skf = StratifiedKFold(n_splits=GNN_N_FOLDS, shuffle=True, random_state=42)
gnn_fold_results = []
best_gnn_model = None
best_gnn_acc   = -1

for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(gnn_pairs)), gnn_labels)):
    print(f'\nFold {fold+1}/{GNN_N_FOLDS}')
    train_pairs  = [gnn_pairs[i] for i in train_idx]
    val_pairs    = [gnn_pairs[i] for i in val_idx]
    train_labels = gnn_labels[train_idx]
    val_labels   = gnn_labels[val_idx]

    model = DDIGNNModel().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
    history   = {'loss': [], 'val_acc': []}

    for epoch in range(GNN_N_EPOCHS):
        loss = gnn_train_epoch(model, optimizer, train_pairs, train_labels)
        acc, _, _ = gnn_evaluate(model, val_pairs, val_labels)
        scheduler.step()
        history['loss'].append(loss)
        history['val_acc'].append(acc)
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | Loss: {loss:.4f} | Val Acc: {acc:.3f}')

    final_acc, final_preds, final_probs = gnn_evaluate(model, val_pairs, val_labels)
    gnn_fold_results.append({
        'fold': fold+1, 'acc': final_acc,
        'preds': final_preds, 'probs': final_probs,
        'true': val_labels, 'history': history
    })
    print(f'  Final Val Acc: {final_acc:.3f}')

    if final_acc > best_gnn_acc:
        best_gnn_acc   = final_acc
        best_gnn_model = model

mean_acc = np.mean([r['acc'] for r in gnn_fold_results])
std_acc  = np.std([r['acc'] for r in gnn_fold_results])
print(f'\nGNN Mean Accuracy: {mean_acc:.3f} ± {std_acc:.3f}')
ddi_gnn = best_gnn_model  # keep best fold


In [ ]:
# ─── 2.7  GNN Training Curves ────────────────────────────────────────────────
from sklearn.metrics import classification_report

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2196F3', '#4CAF50', '#FF5722']

for r in gnn_fold_results:
    c = colors[r['fold'] - 1]
    axes[0].plot(r['history']['loss'],    label=f'Fold {r["fold"]}', color=c)
    axes[1].plot(r['history']['val_acc'], label=f'Fold {r["fold"]}', color=c)

axes[0].set_title('Training Loss');      axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('DDI GNN — Training Curves', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/ddi_gnn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification report on last fold
last = gnn_fold_results[-1]
print(f'Fold {last["fold"]} Classification Report:')
print(classification_report(last['true'], last['preds'], target_names=gnn_class_names, zero_division=0))


---
# Save All Models to Google Drive


In [ ]:
# ─── Save DDI RF Model ────────────────────────────────────────────────────────
# import shutil

# with open(os.path.join(SAVE_DIR, 'ddi_multiclass_model.pkl'), 'wb') as f:
#     pickle.dump(ddi_model, f)
# with open(os.path.join(SAVE_DIR, 'ddi_label_encoder.pkl'), 'wb') as f:
#     pickle.dump(le, f)
# print('RF model saved.')

# ─── Save DDI GNN Model ───────────────────────────────────────────────────────
torch.save(ddi_gnn.state_dict(), os.path.join(SAVE_DIR, 'ddi_gnn_weights.pt'))

gnn_cfg = {
    'n_classes':      N_GNN_CLASSES,
    'hidden':         GNN_HIDDEN,
    'embed':          GNN_EMBED,
    'node_feat_dim':  GNN_NODE_FEAT_DIM,
    'fp_dim':         GNN_FP_DIM,
    'atom_types':     GNN_ATOM_TYPES,
    'fold_accuracies': [r['acc'] for r in gnn_fold_results],
    'mean_cv_acc':    float(np.mean([r['acc'] for r in gnn_fold_results])),
}
with open(os.path.join(SAVE_DIR, 'ddi_gnn_config.json'), 'w') as f:
    json.dump(gnn_cfg, f)

with open(os.path.join(SAVE_DIR, 'ddi_gnn_label_encoder.pkl'), 'wb') as f:
    pickle.dump(gnn_le, f)

# Copy JSON maps + plots
for fn in ['ddi_label_map.json', 'ddi_gnn_label_map.json']:
    shutil.copy(f'/content/{fn}', os.path.join(SAVE_DIR, fn))

for png in ['ddi_class_distribution.png', 'ddi_metrics_bar.png',
            'ddi_per_class_f1.png', 'ddi_confusion_matrix.png',
            'ddi_gnn_training_curves.png']:
    src = f'/content/{png}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, png))

print('\n=== Saved to Google Drive ===')
print('  ddi_multiclass_model.pkl   — 89-class Random Forest')
print('  ddi_label_encoder.pkl      — RF LabelEncoder')
print('  ddi_label_map.json         — {id: description}')
print('  ddi_gnn_weights.pt         — GNN state dict (best CV fold)')
print('  ddi_gnn_config.json        — GNN architecture + CV metrics')
print('  ddi_gnn_label_encoder.pkl  — GNN LabelEncoder (top-5 classes)')
print('  ddi_gnn_label_map.json     — {0-4: description}')
print(f'\nAll files: {sorted(os.listdir(SAVE_DIR))}')
